In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import pandas as pd
 
DB = "hospital_blackhole"
spark.sql(f"USE {DB}")


In [0]:
# STEP 1: Quality gates on Bronze (DLT-equivalent expectations)
# ============================================================
 
# COMMAND ----------
df_hmis_raw = spark.table(f"{DB}.bronze_hmis_referrals")
df_fac_raw  = spark.table(f"{DB}.bronze_facility_master")
df_pmjay    = spark.table(f"{DB}.bronze_pmjay_claims")
df_nfhs     = spark.table(f"{DB}.bronze_nfhs_baseline")
df_dist     = spark.table(f"{DB}.bronze_district_master")
 
# Quality checks
def check_quality(df, name, rules):
    total = df.count()
    for rule_name, condition in rules:
        fails = df.filter(~condition).count()
        pct = round(100 * fails / total, 2)
        status = "✅" if fails == 0 else ("⚠️" if pct < 5 else "❌")
        print(f"  {status} {name}.{rule_name}: {fails} failures ({pct}%)")
    return df
 
print("Quality checks:")
check_quality(df_hmis_raw, "hmis", [
    ("no_null_district",   F.col("district_id").isNotNull()),
    ("no_null_disease",    F.col("disease_category").isNotNull()),
    ("positive_cases",     F.col("cases_diagnosed_phc") > 0),
    ("dropout_consistent", F.col("total_dropout") >= 0),
    ("cascade_monotonic",  F.col("cases_diagnosed_phc") >= F.col("cases_referred")),
])
 
# Apply filters
df_hmis = df_hmis_raw.filter(
    F.col("district_id").isNotNull() &
    F.col("disease_category").isNotNull() &
    (F.col("cases_diagnosed_phc") > 0) &
    (F.col("cases_referred") <= F.col("cases_diagnosed_phc"))
)
print(f"\nHMIS records after quality gate: {df_hmis.count():,}")


In [0]:
# STEP 2: Build silver_patient_journeys
# One row per district × disease × month with all funnel metrics
# ============================================================
 
# COMMAND ----------
# Compute dropout rates at each stage
df_journeys = df_hmis \
    .withColumn("report_date", F.to_date("report_month")) \
    .withColumn("month_num",   F.month("report_date")) \
    .withColumn("year_num",    F.year("report_date")) \
    .withColumn("is_monsoon",  F.col("month_num").isin([6,7,8,9])) \
    .withColumn(
        "stage1_dropout_rate",
        F.round(F.col("dropout_stage1") / F.greatest(F.col("cases_diagnosed_phc"), F.lit(1)), 4)
    ) \
    .withColumn(
        "stage2_dropout_rate",
        F.round(F.col("dropout_stage2") / F.greatest(F.col("cases_referred"), F.lit(1)), 4)
    ) \
    .withColumn(
        "stage3_dropout_rate",
        F.round(F.col("dropout_stage3") / F.greatest(F.col("cases_chc_attended"), F.lit(1)), 4)
    ) \
    .withColumn(
        "stage4_dropout_rate",
        F.round(F.col("dropout_stage4") / F.greatest(F.col("cases_specialist_ref"), F.lit(1)), 4)
    ) \
    .withColumn(
        "stage5_dropout_rate",
        F.round(F.col("dropout_stage5") / F.greatest(F.col("cases_tertiary"), F.lit(1)), 4)
    ) \
    .withColumn(
        "stage6_dropout_rate",
        F.round(F.col("dropout_stage6") / F.greatest(F.col("cases_treatment_started"), F.lit(1)), 4)
    ) \
    .withColumn(
        "overall_dropout_rate",
        F.round(F.col("total_dropout") / F.greatest(F.col("cases_diagnosed_phc"), F.lit(1)), 4)
    ) \
    .withColumn(
        "completion_rate",
        F.round(F.col("cases_completed") / F.greatest(F.col("cases_diagnosed_phc"), F.lit(1)), 4)
    ) \
    .withColumn(
        "critical_disease",
        F.col("disease_category").isin(["Cancer","Tuberculosis","Mental Health"])
    ) \
    .withColumn("processed_at", F.current_timestamp())
 
# Rolling 3-month avg dropout (window per district-disease)
window_rolling = Window \
    .partitionBy("district_id","disease_category") \
    .orderBy(F.unix_date("report_date")) \
    .rowsBetween(-2, 0)
 
df_journeys = df_journeys \
    .withColumn(
        "rolling_3m_dropout",
        F.round(F.avg("overall_dropout_rate").over(window_rolling), 4)
    ) \
    .withColumn(
        "dropout_trending_up",
        F.col("overall_dropout_rate") > F.lag("overall_dropout_rate", 1).over(
            Window.partitionBy("district_id","disease_category").orderBy("report_date")
        )
    )


In [0]:
 
df_journeys.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .partitionBy("state_code","report_year") \
    .saveAsTable(f"{DB}.silver_patient_journeys")
print(f"✅ silver_patient_journeys: {df_journeys.count():,} rows")


In [0]:
# STEP 3: Build silver_dropout_events
# One row per district × disease × STAGE — the pivot table
# that powers the heatmap
# ============================================================
 
# COMMAND ----------
# Aggregate by district + disease — latest 12 months
cutoff = "2023-01-01"
df_agg = df_journeys.filter(F.col("report_date") >= cutoff)
 
df_by_stage = df_agg.groupBy("district_id","district_name","state","disease_category") \
    .agg(
        F.round(F.avg("stage1_dropout_rate") * 100, 1).alias("stage1_pct"),
        F.round(F.avg("stage2_dropout_rate") * 100, 1).alias("stage2_pct"),
        F.round(F.avg("stage3_dropout_rate") * 100, 1).alias("stage3_pct"),
        F.round(F.avg("stage4_dropout_rate") * 100, 1).alias("stage4_pct"),
        F.round(F.avg("stage5_dropout_rate") * 100, 1).alias("stage5_pct"),
        F.round(F.avg("stage6_dropout_rate") * 100, 1).alias("stage6_pct"),
        F.round(F.avg("overall_dropout_rate") * 100, 1).alias("overall_dropout_pct"),
        F.round(F.avg("completion_rate") * 100, 1).alias("completion_pct"),
        F.sum("cases_diagnosed_phc").alias("total_diagnosed"),
        F.sum("cases_completed").alias("total_completed"),
        F.sum("total_dropout").alias("total_dropped"),
        F.first("vulnerability_idx").alias("vulnerability_idx"),
    )
 
df_by_stage.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.silver_dropout_events")
print(f"✅ silver_dropout_events: {df_by_stage.count():,} rows")
 


In [0]:
# STEP 4: Build silver_facility_capacity
# Specialist availability as a feature for BHS
# ============================================================
 
# COMMAND ----------
# For each district: aggregate facility capacity signals
df_fac_dist = df_fac_raw.groupBy("district_id") \
    .agg(
        F.sum("beds_functional").alias("total_functional_beds"),
        F.sum("specialists_functional").alias("total_specialists"),
        F.sum("specialists_sanctioned").alias("specialists_sanctioned"),
        F.avg("vacancy_rate").alias("avg_vacancy_rate"),
        F.count(F.when(F.col("facility_type") == "District Hospital", 1)).alias("n_dh"),
        F.count(F.when(F.col("facility_type") == "Medical College", 1)).alias("n_mc"),
        F.count(F.when(F.col("facility_type") == "Community HC", 1)).alias("n_chc"),
        F.count(F.when(F.col("facility_type") == "Primary HC", 1)).alias("n_phc"),
        F.avg("travel_hours_to_dh").alias("avg_travel_hrs_to_dh"),
        F.min("travel_hours_to_dh").alias("min_travel_hrs_to_dh"),
    )
 
# Specialty availability rates (from facility vacancies)
SPECIALTIES = ["Oncology","Pulmonology","Cardiology","Endocrinology",
               "Haematology","Gynaecology","Psychiatry","Orthopaedics"]
 
import random as rnd
rnd.seed(77)
 
spec_rows = []
for row in df_fac_dist.collect():
    vac = row["avg_vacancy_rate"] or 0.3
    for spec in SPECIALTIES:
        # Harder-to-fill specialties have worse vacancy
        base_avail = max(0.05, 1.0 - vac - rnd.uniform(0, 0.3))
        if spec in ["Oncology","Psychiatry","Haematology"]:
            base_avail *= 0.5  # chronic shortage
        spec_rows.append({
            "district_id"       : row["district_id"],
            "specialty"         : spec,
            "availability_pct"  : round(max(0.05, min(1.0, base_avail)) * 100, 1),
            "requirement_pct"   : 100.0,
            "gap_pct"           : round(max(0, (1.0 - base_avail) * 100), 1),
        })
 
df_spec = spark.createDataFrame(pd.DataFrame(spec_rows))
df_cap = df_fac_dist.join(df_spec.groupBy("district_id").agg(
    F.avg("availability_pct").alias("avg_specialty_availability_pct"),
    F.min("availability_pct").alias("min_specialty_availability_pct"),
), on="district_id", how="left")
 
df_cap.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.silver_facility_capacity")
df_spec.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.silver_specialty_gaps")
print(f"✅ silver_facility_capacity: {df_cap.count()} districts")


In [0]:
# STEP 5: Build silver_geo_barriers
# Travel time as a geographic dropout predictor
# ============================================================
 
# COMMAND ----------
# Join district travel times with dropout rates
df_geo = df_dist.select("district_id","district_name","state","population",
                         "vulnerability_idx","tribal_pct","lat","lon") \
    .join(
        df_fac_dist.select("district_id","avg_travel_hrs_to_dh","min_travel_hrs_to_dh"),
        on="district_id", how="left"
    ) \
    .join(
        df_by_stage.groupBy("district_id").agg(
            F.round(F.avg("overall_dropout_pct"), 1).alias("avg_dropout_pct"),
        ),
        on="district_id", how="left"
    )
 
# Geographic barrier score (0–100): longer travel = higher barrier
max_travel = df_geo.agg(F.max("avg_travel_hrs_to_dh")).collect()[0][0] or 8.0
df_geo = df_geo.withColumn(
    "geo_barrier_score",
    F.round(
        F.col("avg_travel_hrs_to_dh") / max_travel * 100, 1
    )
).withColumn("processed_at", F.current_timestamp())
 
df_geo.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.silver_geo_barriers")
print(f"✅ silver_geo_barriers: {df_geo.count()} districts")


In [0]:
# STEP 6: silver_survival_features
# Prepare features for Kaplan-Meier + XGBoost model
# ============================================================
 
# COMMAND ----------
df_events_raw = spark.table(f"{DB}.bronze_patient_events")
 
# Last event per patient = their terminal state
window_last = Window.partitionBy("patient_id").orderBy(F.col("event_date").desc())
df_terminal = df_events_raw.withColumn("rn", F.row_number().over(window_last)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")
 
# First event per patient = diagnosis
window_first = Window.partitionBy("patient_id").orderBy("event_date")
df_diag = df_events_raw.filter(F.col("event_type") == "DIAGNOSED") \
    .withColumn("rn", F.row_number().over(window_first)) \
    .filter(F.col("rn") == 1) \
    .select("patient_id", F.col("event_date").alias("diagnosis_date")) \
    .drop("rn")
 
# Survival features
df_surv = df_terminal.join(df_diag, on="patient_id", how="left") \
    .withColumn(
        "survival_days",
        F.datediff(F.col("event_date").cast("date"), F.col("diagnosis_date").cast("date"))
    ) \
    .withColumn("event_occurred", F.col("event_type") == "TREATMENT_COMPLETED") \
    .withColumn("is_censored",    ~F.col("event_occurred")) \
    .withColumn(
        "dropout_stage",
        F.when(F.col("event_type") == "DROPPED_OUT",
               F.coalesce(F.col("dropout_stage"), F.lit("Unknown")))
         .otherwise("Completed")
    ) \
    .select(
        "patient_id","district_id","state","disease",
        "survival_days","event_occurred","is_censored",
        "dropout_stage","vulnerability",
        "diagnosis_date"
    ) \
    .withColumn("survival_days", F.greatest(F.col("survival_days"), F.lit(1)))
 
df_surv.write.format("delta").mode("overwrite").option("overwriteSchema","true") \
    .saveAsTable(f"{DB}.silver_survival_features")
print(f"✅ silver_survival_features: {df_surv.count():,} patient journeys")
 
# COMMAND ----------
print("\n" + "="*55)
print("SILVER LAYER COMPLETE")
print("="*55)
for tbl in ["silver_patient_journeys","silver_dropout_events",
            "silver_facility_capacity","silver_geo_barriers",
            "silver_survival_features"]:
    cnt = spark.table(f"{DB}.{tbl}").count()
    print(f"  {tbl:<35} {cnt:>8,} rows")
print("="*55)